In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pycocotools
!pip install opencv-python

In [ ]:
from pycocotools.coco import COCO
import numpy as np
import cv2
import os
from tqdm import tqdm

base_path = r'/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017'

# Split-specific configs
splits = {
    "train": {
        "ann_file": os.path.join(
            base_path,
            "annotations",
            "instances_train2017.json"
        )
    },
    "val": {
        "ann_file": os.path.join(
            base_path,
            "annotations",
            "instances_val2017.json"
        )
    },
}

for split_name, cfg in splits.items():
    print(f"\n=== Processing {split_name} ===")

    # Output directory for this split
    masks_output_dir = os.path.join(base_path, f"mask/masks_{split_name}")
    os.makedirs(masks_output_dir, exist_ok=True)
    print(f"Saving masks to: {masks_output_dir}")

    # Load annotations for the split
    ann_path = cfg["ann_file"]
    coco = COCO(ann_path)   # annToMask etc. [web:1][web:6]
    img_ids = coco.getImgIds()
    print(f"Total {split_name} images: {len(img_ids)}")

    # Create masks
    for img_id in tqdm(img_ids, desc=f"Creating {split_name} masks"):
        try:
            img_info = coco.loadImgs(img_id)[0]
            ann_ids = coco.getAnnIds(imgIds=img_id)
            anns = coco.loadAnns(ann_ids)

            # Empty binary mask
            mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)

            # Combine all instance masks for this image
            for ann in anns:
                mask = np.bitwise_or(mask, coco.annToMask(ann).astype(np.uint8))

            # Convert to 0/255
            mask = (mask > 0).astype(np.uint8) * 255

            # Save with matching filename
            mask_filename = img_info['file_name'].replace('.jpg', '_mask.png')
            mask_path = os.path.join(masks_output_dir, mask_filename)

            # Note: function is imwrite, not inwrite [web:10]
            cv2.imwrite(mask_path, mask)

        except Exception as e:
            print(f"Error processing image {img_id}: {str(e)}")

    # Verify masks were created
    mask_count = len(os.listdir(masks_output_dir))
    print(f"✓ Verified: {mask_count} mask files created in {masks_output_dir}")

print("✓ Train and val masks created successfully!")


In [ ]:
## Files count in a folder
import os
folder_path = r'/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017/mask/masks_train'
file_count = len(os.listdir(folder_path))
print(f"Number of files in the folder: {file_count}")